# Import packages

In [94]:
import pandas as pd
from datasets import load_dataset
import numpy as np

from collections import Counter

import nltk
from nltk.tokenize import word_tokenize
nltk.download('punkt_tab')

from sklearn.metrics import classification_report


[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Fazz\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


# Load and split/filter data

Load the dataset, and then from languages provided by assignmenttext, filter it from the full dataset.


df_train/validation_filtered contains the dataset with all 3 lanauges.<br>
df_train/validation_ar/ko/te contains the dataset split for each lanauges

In [2]:
dataset = load_dataset("coastalcph/tydi_xor_rc")


df_train = dataset["train"].to_pandas()
df_validation = dataset["validation"].to_pandas()

# Selected languages
langlst = ['ko','ar','te']

# Training and Validation data for selected languages
df_train_filtered = df_train[df_train['lang'].isin(langlst)]
df_validation_filtered = df_validation[df_validation['lang'].isin(langlst)]
 
#Training and Validation data for each language
df_train_ar = df_train[(df_train['lang'] == "ar")]
df_train_ko = df_train[(df_train['lang'] == "ko")]
df_train_te = df_train[(df_train['lang'] == "te")]

df_validation_ar = df_validation[(df_validation['lang'] == "ar")]
df_validation_ko = df_validation[(df_validation['lang'] == "ko")]
df_validation_te = df_validation[(df_validation['lang'] == "te")]


# Statistics on the dataset

For each language and train/validation split, anylyze numbers of examples, answerable/unanswerable percentage, question median length (in tokens), question inter quartile range (in tokens), context median length (in tokens), and context inter quartile range (in tokens). <br> <i>All rounded to 1 decimal</i>.

Questions and Context are tokenized using the nltk tokenizer, splitting at whitespace

In [32]:
def get_statistics(df, split, language):
    question_lengths = []
    context_lengths = []

    for question in df["question"]:
        question_lengths.append(len(word_tokenize(question)))
    
    for context in df["context"]:
        context_lengths.append(len(word_tokenize(context)))

    answerable_n = len(df[df["answerable"] == True])
    unanswerable_n = len(df[df["answerable"] == False])

    return {
            "split": split,
            "language": language,
            "n": len(df),
            "answerable_%": 100 * answerable_n / len(df),
            "unanswerable_%": 100 * unanswerable_n / len(df),
            "question_median": np.median(question_lengths),
            "question_IQR": np.percentile(question_lengths, 75) - np.percentile(question_lengths, 25),
            "context_median": np.median(context_lengths),
            "context_IQR": np.percentile(context_lengths, 75) - np.percentile(context_lengths, 25)
        }


statistics = pd.DataFrame([
    get_statistics(df_train_ar, "train", "ar"),
    get_statistics(df_train_ko, "train", "ko"),
    get_statistics(df_train_te, "train", "te"),
    get_statistics(df_validation_ar, "validation", "ar"),
    get_statistics(df_validation_ko, "validation", "ko"),
    get_statistics(df_validation_te, "validation", "te")
])

statistics.round(1)


,split,language,n,answerable_%,unanswerable_%,question_median,question_IQR,context_median,context_IQR
0,train,ar,2558,90.0,10.0,6.0,3.0,103.0,80.0
1,train,ko,2422,97.4,2.6,6.0,2.0,99.0,73.0
2,train,te,1355,96.7,3.3,6.0,3.0,93.0,71.0
3,validation,ar,415,87.5,12.5,6.0,3.0,103.0,77.0
4,validation,ko,356,94.7,5.3,6.0,2.0,97.5,85.2
5,validation,te,384,75.8,24.2,7.0,2.0,120.0,76.0


### And for overall filtered dataset

In [33]:
def get_answerable(df, split):
    answerable_n = len(df[df["answerable"] == True])
    unanswerable_n = len(df[df["answerable"] == False])

    return {
        "split": split,
        "n": len(df),
        "answerable_%": 100 * answerable_n / len(df),
        "unanswerable_%": 100 * unanswerable_n / len(df)
    }



overall_rows = pd.DataFrame([
    get_answerable(df_train_filtered, "train"),
    get_answerable(df_validation_filtered, "validation")
])

overall_rows.round(1)


,split,n,answerable_%,unanswerable_%
0,train,6335,94.3,5.7
1,validation,1155,85.8,14.2


# Data quality
For each language and split, check for missing values and duplicate question/context pairs.

In [19]:
data_quality_rows = []

dataframes = [
    ("train", "ar", df_train_ar),
    ("train", "ko", df_train_ko),
    ("train", "te", df_train_te),
    ("validation", "ar", df_validation_ar),
    ("validation", "ko", df_validation_ko),
    ("validation", "te", df_validation_te)
]

for split, language, df in dataframes:
    missing_values = df.isna().sum().sum()
    duplicate_pairs = df[["question", "context"]].duplicated().sum()

    data_quality_rows.append({
        "split": split,
        "language": language,
        "missing_values": missing_values,
        "duplicate_question_context_pairs": duplicate_pairs
    })

pd.DataFrame(data_quality_rows)

,split,language,missing_values,duplicate_question_context_pairs
0,train,ar,2558,0
1,train,ko,2422,10
2,train,te,1305,0
3,validation,ar,415,0
4,validation,ko,356,0
5,validation,te,284,0


# Most common tokens

Analyze the most common tokens in the training split, <i> With questionmarks excluded</i>.

In [85]:
def get_most_common_tokens(df, language):
    tokens = []

    for question in df["question"]:
        for token in word_tokenize(question):
            if token not in ["?", "؟"]:
                tokens.append(token)

    top5 = Counter(tokens).most_common(5)

    return pd.DataFrame([
        {
            "language": language,
            "token": token,
            "count": count
        }
        for token, count in top5
    ])

display(get_most_common_tokens(df_train_ar, "ar"))
display(get_most_common_tokens(df_train_ko, "ko"))
display(get_most_common_tokens(df_train_te, "te"))


,language,token,count
0,ar,في,593
1,ar,من,587
2,ar,متى,536
3,ar,ما,443
4,ar,هو,349


,language,token,count
0,ko,가장,527
1,ko,무엇인가,497
2,ko,언제,336
3,ko,몇,234
4,ko,어디인가,228


,language,token,count
0,te,ఎవరు,274
1,te,ఏది,192
2,te,ఎన్ని,165
3,te,ఎప్పుడు,154
4,te,ఏ,142


# Majority Baseline, and Rule-based Baseline

Majority baseline implements: analyzing number of answerable and unanswerable, and checks for what the majority is.

Rule-based baseline implements: A rule-based determination of whether a question is answerable or not. Rule goes as:
- No numbers = <b> answerable </b>
- Same number in question and context = <b> answerable </b>
- Number in question, but not in context = <b> unanswerable </b>

In [92]:
answerable_n = len(df_train_filtered[df_train_filtered["answerable"] == True])
unanswerable_n = len(df_train_filtered[df_train_filtered["answerable"] == False])

if answerable_n >= unanswerable_n:
    majority_label = True
else:
    majority_label = False

print("Answerable:", answerable_n)
print("Unanswerable:", unanswerable_n)
print("Majority prediction:", majority_label)


Answerable: 5972
Unanswerable: 363
Majority prediction: True


In [93]:
# Rule-based baseline: no question numbers, or at least one matching number.
import re

def extract_numbers(text):
    # int() converts each Unicode decimal digit to its ASCII equivalent.
    return {''.join(str(int(digit)) for digit in number)
            for number in re.findall(r'\d+', text)}

def rule_answerable(question, context):
    question_numbers = extract_numbers(question)
    context_numbers = extract_numbers(context)
    return not question_numbers or bool(question_numbers & context_numbers)

rule_predictions = [
    rule_answerable(question, context)
    for question, context in zip(
        df_validation_filtered['question'], df_validation_filtered['context']
    )
]

bin_list = [1 if elm else 0 for elm in rule_predictions]

print(sum(bin_list))
print(sum(bin_list)/len(bin_list))

1083
0.9376623376623376


# Scoring of Baseline predictions

Scoring the prediction models, using <i>Accuracy</i>, <i>Precision</i>, <i>recall</i>, and <i>Macro_F1</i>, for both majority baseline, and the rule-based baseline.

In [100]:
evaluation = df_validation_filtered.copy()
evaluation["majority"] = majority_label
evaluation["number_overlap"] = rule_predictions

results = []

for language in ["all", "ar", "ko", "te"]:
    group = evaluation if language == "all" else evaluation[evaluation["lang"] == language]

    for baseline in ["majority", "number_overlap"]:
        report = classification_report(
            group["answerable"],
            group[baseline],
            labels=[False, True],
            target_names=["unanswerable", "answerable"],
            output_dict=True,
            zero_division=0
        )

        results.append({
            "language": language,
            "baseline": baseline,
            "n": len(group),
            "accuracy": report["accuracy"],
            "precision": report["macro avg"]["precision"],
            "recall": report["macro avg"]["recall"],
            "macro_f1": report["macro avg"]["f1-score"]
        })

baseline_results = pd.DataFrame(results)

display(baseline_results.round(3))


,language,baseline,n,accuracy,precision,recall,macro_f1
0,all,majority,1155,0.858,0.429,0.500,0.462
1,all,number_overlap,1155,0.815,0.506,0.503,0.495
2,ar,majority,415,0.875,0.437,0.500,0.467
3,ar,number_overlap,415,0.860,0.500,0.500,0.479
4,ko,majority,356,0.947,0.473,0.500,0.486
5,ko,number_overlap,356,0.846,0.485,0.471,0.476
6,te,majority,384,0.758,0.379,0.500,0.431
7,te,number_overlap,384,0.737,0.556,0.519,0.498


# Error Analysis

Examples where rule-based baseline predicted wrong, producing false positives/negatives. 

In [101]:
false_positives = evaluation[evaluation['number_overlap'] & ~evaluation['answerable']]
false_negatives = evaluation[~evaluation['number_overlap'] & evaluation['answerable']]

print('Majority false positives:', sum(evaluation['majority'] & ~evaluation['answerable']))
print('Majority false negatives:', sum(~evaluation['majority'] & evaluation['answerable']))
print('Number-overlap false positives:', len(false_positives))
print('Number-overlap false negatives:', len(false_negatives))

columns = ['lang', 'question', 'context', 'answer']
display(false_positives[columns].head(5))
display(false_negatives[columns].head(5))


Majority false positives: 164
Majority false negatives: 0
Number-overlap false positives: 153
Number-overlap false negatives: 61


,lang,question,context,answer
364,ko,시차는 중력과 관련이 있는가?,Gravitational time dilation was first describe...,no
368,ko,맹장 없이 살 수 있을까?,"In modern humans, the vermiform appendix is a ...",no
369,ko,무한은 공식이 있을까?,"In mathematics, ""infinity"" is often treated as...",no
398,ko,핵융합은 상용화 가능성이 있는가?,Several experimental nuclear fusion reactors a...,no
417,ko,화성의 대기에 인간이 살 수 있는가?,It is unforeseen how the first human landing o...,no


,lang,question,context,answer
16,te,2010 నాటికి భారతదేశంలో క్రైస్తవులు ఎక్కువ ఉండే...,Even though Christians are a significant minor...,"Meghalaya, Mizoram, and Nagaland"
29,te,2018 వరకు బైబిలు ను ఎన్ని భాషలలో అనువదించారు?,The Bible has been translated into many langua...,"3,312 languages"
46,te,2019 వరకు బైబులును ఎన్ని భాషలలో అనువదించారు?,The Bible has been translated into many langua...,670
69,te,2017లో ఆలివ్ నూనె ఉత్పత్తి లో ప్రథమ స్థానంలో ఉ...,Olive trees have been grown around the Mediter...,Spain
70,te,2017లో ఆలివ్ నూనె ఉత్పత్తి లో ప్రథమ స్థానంలో ఉ...,"In 2016/17 period, world production of virgin ...",Spain
